In [1]:
import os
import json
import shutil
import pandas as pd


In [2]:
class SituationSaver:
	def __init__(self, language: str, base_dir: str = "."):
		self.language = language

		self.base_dir = base_dir
		self.localization_dir = os.path.join(base_dir, "localization")

		self.dialogue_dir = os.path.join(self.localization_dir, "dialogue")
		self.modified_dir = os.path.join(self.dialogue_dir, "modified", language)
		self.changed_dir = os.path.join(self.dialogue_dir, "changed", language)
		self.active_dir = os.path.join(self.dialogue_dir, "active", language)

		self.structure_dir = os.path.join(self.localization_dir, "structure", "modified")

		self.data_dir = os.path.join(base_dir, "data")
		self.situations_dir = os.path.join(self.data_dir, "processed")

		self.metadata_path = os.path.join(self.data_dir, "metadata.json")

		with open(self.metadata_path, "r", encoding="utf-8") as f:
			self.metadata = json.load(f)

		self.state = {}

	def _load_situation_config(self, situation_number: int):
		config = self.metadata.get(str(situation_number))
		if not config:
			raise ValueError(f"Situation {situation_number} not found in metadata.")
		return config

	def _load_df(self, situation_number: int):
		path = os.path.join(
			self.situations_dir,
			f"situation_{situation_number}.csv"
		)
		return pd.read_csv(path, encoding="utf-8")

	def process_situation(self, number: int):
		config = self._load_situation_config(number)

		keys = config["object"].split(".")
		file = config["file"]
		context = config["context"]
		extra_nodes = config["extra_nodes"]

		df = self._load_df(number)

		structure_path = os.path.join(self.structure_dir, file)

		with open(structure_path, "r", encoding="utf-8") as f:
			structure = json.load(f)

		if file not in self.state:
			language_path = os.path.join(self.modified_dir, file)
			with open(language_path, "r", encoding="utf-8") as f:
				self.state[file] = json.load(f)
		
		data = self.state[file]

		obj = data
		struct = structure

		for key in keys[:-1]:
			if key not in obj:
				obj[key] = {}

			obj = obj[key]
			struct = struct[key]

		parent_obj = obj
		parent_struct = struct

		target_key = keys[-1]
		target_struct = parent_struct[target_key]

		groups = {
			choice: group["text_clean"].tolist()
			for choice, group in df.groupby("choice")
		}

		responses = [
			{"text": groups[choice["next"]]}
			for choice in target_struct["choices"]
		]

		node = {
			"context": context,
			"responses": responses
		}

		new_parent = {}

		for key in parent_struct:
			if key == target_key:
				new_parent[key] = node
			elif key in extra_nodes:
				new_parent[key] = extra_nodes[key].copy()
			elif key in parent_obj:
				new_parent[key] = parent_obj[key]

		parent_obj.clear()
		parent_obj.update(new_parent)

		return data

	def process_multiple(self, numbers: list[int]):
		results = {}

		for n in numbers:
			print(f"Processing situation {n}...")
			results[n] = self.process_situation(n)

		return results

	def save(self):
		os.makedirs(self.changed_dir, exist_ok=True)

		for file, data in self.state.items():
			out_path = os.path.join(self.changed_dir, file)

			os.makedirs(os.path.dirname(out_path), exist_ok=True)

			with open(out_path, "w", encoding="utf-8") as f:
				json.dump(data, f, ensure_ascii=False, indent=4)

			print(f"Saved: {out_path}")

	def build_final(self):
		if os.path.exists(self.active_dir):
			shutil.rmtree(self.active_dir)

		shutil.copytree(self.modified_dir, self.active_dir)

		for file, data in self.state.items():
			out_path = os.path.join(self.active_dir, file)

			os.makedirs(os.path.dirname(out_path), exist_ok=True)

			with open(out_path, "w", encoding="utf-8") as f:
				json.dump(data, f, ensure_ascii=False, indent=4)

			print(f"Updated: {out_path}")


In [3]:
processor = SituationSaver("es")

# data = processor.process_situation(2)
processor.process_multiple([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12])
processor.save()
# pprint(data)


Processing situation 1...
Processing situation 2...
Processing situation 3...
Processing situation 4...
Processing situation 5...
Processing situation 6...
Processing situation 7...
Processing situation 8...
Processing situation 9...
Processing situation 10...
Processing situation 11...
Processing situation 12...
Saved: .\localization\dialogue\changed\es\scene1/scene1Classroom.json
Saved: .\localization\dialogue\changed\es\scene1/scene1Bedroom1.json
Saved: .\localization\dialogue\changed\es\scene1/scene1Bedroom2.json
Saved: .\localization\dialogue\changed\es\scene2/scene2Break.json
Saved: .\localization\dialogue\changed\es\scene3/scene3Bedroom.json
Saved: .\localization\dialogue\changed\es\scene4/scene4Backyard.json
Saved: .\localization\dialogue\changed\es\scene4/scene4Garage.json
Saved: .\localization\dialogue\changed\es\scene4/scene4Bedroom.json
Saved: .\localization\dialogue\changed\es\scene6/routeA/scene6BedroomRouteA1.json
Saved: .\localization\dialogue\changed\es\scene6/routeB/s

In [4]:
processor.build_final()


Updated: .\localization\dialogue\active\es\scene1/scene1Classroom.json
Updated: .\localization\dialogue\active\es\scene1/scene1Bedroom1.json
Updated: .\localization\dialogue\active\es\scene1/scene1Bedroom2.json
Updated: .\localization\dialogue\active\es\scene2/scene2Break.json
Updated: .\localization\dialogue\active\es\scene3/scene3Bedroom.json
Updated: .\localization\dialogue\active\es\scene4/scene4Backyard.json
Updated: .\localization\dialogue\active\es\scene4/scene4Garage.json
Updated: .\localization\dialogue\active\es\scene4/scene4Bedroom.json
Updated: .\localization\dialogue\active\es\scene6/routeA/scene6BedroomRouteA1.json
Updated: .\localization\dialogue\active\es\scene6/routeB/scene6LunchRouteB.json
